# Exercise 2

This notebook now follows Problem 2 of the exam brief: evaluate entanglement properties of a neural quantum state. The focus is not variational training yet, but rather the state representation, sampling logic, SWAP-based Renyi-2 estimation, and the behavior of randomly initialized networks under disorder averaging.


In [ ]:
from notebook_bootstrap import bootstrap_notebook

bootstrap_notebook(enable_x64=True)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from exercise_report_helper import (
    build_output_manifest,
    ensure_report_output_dir,
    save_report_figure,
    save_report_table,
)
from nqs.workflows import run_random_architecture_study

In [ ]:
study_config = {
    'lattice_shape': (2, 2),
    'pbc': False,
    'hamiltonian': 'tfim',
    'J': 1.0,
    'h': 1.0,
    'n_samples': 128,
    'n_discard_per_chain': 32,
    'n_chains': 16,
}

exercise_output_dir = ensure_report_output_dir('exercise_2')
exercise_output_dir

## 2/a Neural Quantum State Representation

We represent the wavefunction as a neural network $\psi_	heta(\sigma)$ that maps a spin configuration $\sigma$ to a complex log-amplitude. For the required architecture families we use:

- `RBM` as the reference compact ansatz;
- `FFNN` as a generic fully connected baseline;
- `CNN` as the explicitly local architecture.

The network interface is shared across architectures, so the downstream sampling and entropy-estimation code depends only on the variational-state API rather than on a specific ansatz implementation.


In [ ]:
architecture_table = pd.DataFrame([
    {'model': 'RBM', 'model_kwargs': {'alpha': 1}, 'role': 'reference compact ansatz'},
    {'model': 'FFNN', 'model_kwargs': {'hidden_dims': (8,)}, 'role': 'fully connected baseline'},
    {'model': 'CNN', 'model_kwargs': {'channels': (4,), 'kernel_size': (2, 2)}, 'role': 'local architecture'},
])
architecture_table

## 2/b Architecture-Agnostic Monte Carlo Sampling

Monte Carlo sampling only needs ratios of probabilities between configurations, so the sampler should work for any architecture that can evaluate $\log \psi_	heta(\sigma)$. In this repository the variational-state object owns the sampler and exposes configuration sampling independently of whether the underlying model is an RBM, FFNN, or CNN.

This separation is the core requirement from the exam: state representation and sampling are coupled only through the log-amplitude interface, not through architecture-specific code paths.


## 2/c Renyi-2 From The SWAP / Local-Estimator Path

For a neural quantum state, the practical route to Renyi-2 is the SWAP estimator. We estimate the purity of $ho_A$ by sampling pairs of configurations from two copies of the same state and evaluating the local SWAP estimator. On small systems we can still compare the sampled result against the exact Renyi-2 obtained from the exact statevector of the same randomly initialized model.


In [ ]:
exercise_2_swap_demo = run_random_architecture_study(
    architecture_configs={'RBM': {'alpha': 1}},
    seeds=(0,),
    entropy_n_independent_runs=8,
    real_amplitude_only=True,
    **study_config,
)

exercise_2_swap_table = exercise_2_swap_demo['entropy_scan_table'].copy()
exercise_2_swap_table[['model', 'subsystem_size', 'exact_renyi2', 'sampled_renyi2', 'estimator_std']]

In [ ]:
swap_figure, swap_axis = plt.subplots(figsize=(7, 4.5))
swap_axis.plot(
    exercise_2_swap_table['subsystem_size'],
    exercise_2_swap_table['exact_renyi2'],
    marker='s',
    label='exact Renyi-2',
)
swap_axis.errorbar(
    exercise_2_swap_table['subsystem_size'],
    exercise_2_swap_table['sampled_renyi2'],
    yerr=exercise_2_swap_table['estimator_std'],
    marker='o',
    capsize=3,
    label='sampled SWAP estimate',
)
swap_axis.set_xlabel('Subsystem size |A|')
swap_axis.set_ylabel('Renyi-2')
swap_axis.set_title('Exercise 2 SWAP estimator on a random RBM state')
swap_axis.grid(alpha=0.25)
swap_axis.legend()
swap_figure.tight_layout()
swap_figure

The estimator noise grows as the subsystem gets larger because the swapped and unswapped amplitudes become harder to estimate from a finite Monte Carlo sample. This is the main practical limitation of the SWAP route for neural quantum states: it is scalable compared with exact reduced-density-matrix methods, but statistical quality degrades as the entangled region grows.


## 2/d Random Initialization, Disorder Averaging, And Parameter Count

Randomly initialized neural quantum states already carry nontrivial entanglement. To obtain a meaningful statement we average over several random seeds and compare the half-partition Renyi-2 across those random initializations. Here we deliberately use a real-amplitude initialization for the random-state study so that the sampled SWAP estimator stays numerically interpretable; this is one concrete initialization choice that directly affects the observed entanglement statistics. The parameter count matters because larger networks can represent more complicated amplitude structure even before training.


In [ ]:
exercise_2_rbm_disorder = run_random_architecture_study(
    architecture_configs={'RBM': {'alpha': 1}},
    seeds=(0, 1, 2, 3),
    entropy_n_independent_runs=8,
    real_amplitude_only=True,
    **study_config,
)

exercise_2_rbm_disorder['summary_table']

In [ ]:
exercise_2_rbm_trials = exercise_2_rbm_disorder['trial_table'].copy()
exercise_2_rbm_trials[['model', 'seed', 'parameter_count', 'half_partition_exact_renyi2', 'half_partition_sampled_renyi2', 'half_partition_sampled_std', 'sampled_minus_exact']]

## 2/e Architecture Comparison Under Random Initialization

We now repeat the random-state study for the fully connected and local architectures. The goal is not to claim a universal scaling law from a tiny system, but to identify qualitative architecture-dependent differences and to see how the Renyi-2 statistics depend on the number of parameters and on locality structure.


In [ ]:
exercise_2_architectures = run_random_architecture_study(
    architecture_configs={
        'RBM': {'alpha': 1},
        'FFNN': {'hidden_dims': (8,)},
        'CNN': {'channels': (4,), 'kernel_size': (2, 2)},
    },
    seeds=(0, 1, 2),
    entropy_n_independent_runs=8,
    real_amplitude_only=True,
    **study_config,
)

exercise_2_architectures['summary_table']

In [ ]:
exercise_2_architecture_scan = exercise_2_architectures['entropy_scan_table'].copy()

architecture_figure, architecture_axis = plt.subplots(figsize=(7.5, 4.5))
for model, group in exercise_2_architecture_scan.groupby('model'):
    ordered = group.sort_values('subsystem_size')
    architecture_axis.errorbar(
        ordered['subsystem_size'],
        ordered['sampled_renyi2'],
        yerr=ordered['estimator_std'],
        marker='o',
        capsize=3,
        label=f'{model} sampled',
    )
architecture_axis.set_xlabel('Subsystem size |A|')
architecture_axis.set_ylabel('Sampled Renyi-2')
architecture_axis.set_title('Exercise 2 disorder-averaged random-state entanglement by architecture')
architecture_axis.grid(alpha=0.25)
architecture_axis.legend()
architecture_figure.tight_layout()
architecture_figure

This small-system study is enough to answer the exam-level qualitative questions: random initializations already produce architecture-dependent entanglement, disorder averaging is necessary to stabilize the conclusions, and the sampled SWAP estimator becomes noisier for larger subsystems even before any training is introduced. On finite samples, especially when the exact entropy is very small, the sampled estimator can even drift slightly below zero; that is a Monte Carlo artifact of the estimator, not a physical negative entropy.

## Export Exercise 2 Artifacts

Persist the random-state summary tables and the two main figures for later report assembly.

In [ ]:
swap_paths = save_report_table(exercise_2_swap_table, 'exercise_2_swap_table', output_dir=exercise_output_dir)
rbm_summary_paths = save_report_table(exercise_2_rbm_disorder['summary_table'], 'exercise_2_rbm_disorder_summary', output_dir=exercise_output_dir)
rbm_trial_paths = save_report_table(exercise_2_rbm_trials, 'exercise_2_rbm_disorder_trials', output_dir=exercise_output_dir)
architecture_summary_paths = save_report_table(exercise_2_architectures['summary_table'], 'exercise_2_architecture_summary', output_dir=exercise_output_dir)
architecture_scan_paths = save_report_table(exercise_2_architecture_scan, 'exercise_2_architecture_scan', output_dir=exercise_output_dir)
swap_figure_path = save_report_figure(swap_figure, 'exercise_2_swap_figure', output_dir=exercise_output_dir)
architecture_figure_path = save_report_figure(architecture_figure, 'exercise_2_architecture_figure', output_dir=exercise_output_dir)

build_output_manifest([
    {'section': 'exercise_2', 'name': 'swap_table', 'path': str(swap_paths['csv'])},
    {'section': 'exercise_2', 'name': 'rbm_disorder_summary', 'path': str(rbm_summary_paths['csv'])},
    {'section': 'exercise_2', 'name': 'rbm_disorder_trials', 'path': str(rbm_trial_paths['csv'])},
    {'section': 'exercise_2', 'name': 'architecture_summary', 'path': str(architecture_summary_paths['csv'])},
    {'section': 'exercise_2', 'name': 'architecture_scan', 'path': str(architecture_scan_paths['csv'])},
    {'section': 'exercise_2', 'name': 'swap_figure', 'path': str(swap_figure_path)},
    {'section': 'exercise_2', 'name': 'architecture_figure', 'path': str(architecture_figure_path)},
])